Question ***1***

In [ ]:
from skimage.feature import hog
from skimage.io import imread
from skimage.transform import resize
from sklearn.model_selection import train_test_split
import numpy as np
import glob
import kagglehub
from skimage.color import rgb2gray
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

In [ ]:
!kaggle datasets download -d ashishsaxena2209/animal-image-datasetdog-cat-and-panda


Dataset URL: https://www.kaggle.com/datasets/ashishsaxena2209/animal-image-datasetdog-cat-and-panda
License(s): unknown
100% 375M/376M [00:18<00:00, 24.0MB/s]
100% 376M/376M [00:18<00:00, 21.2MB/s]


In [ ]:
import zipfile

with zipfile.ZipFile("animal-image-datasetdog-cat-and-panda.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/animal_dataset")


In [ ]:
# Load dataset

path = "/content/animal_dataset/animals/"
animal_images = []

animal_labels = []

for animal_type in ['cats', 'dogs', 'panda']:

  for image_path in glob.glob(f'{path}/{animal_type}/*.jpg'):

    image = imread(image_path)
    image = resize(image, (128, 64)) # Resize images to a fixed size

    # Check if the image is already grayscale
    if len(image.shape) == 2:
        # If grayscale, convert to RGB by stacking the grayscale image 3 times
        image = np.stack((image,) * 3, axis=-1)

    animal_images.append(image)
    animal_labels.append(animal_type)

# Extract HOG features
hog_features = [hog(rgb2gray(image), pixels_per_cell=(8, 8), cells_per_block=(2, 2),
visualize=False) for image in animal_images]

hog_features = np.array(hog_features)

# **TRAINING USING TRANVERSE LEARNING**

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
import shutil  # Import shutil for directory operations

def transfer_learning(base_dir):
    # Dataset Directory
    # Create train and validation directories if they don't exist
    train_dir = os.path.join(base_dir, 'train')
    val_dir = os.path.join(base_dir, 'validation')
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(val_dir, exist_ok=True)

    # Before creating generators, organize data into train/validation
    for animal_type in ['cats', 'dogs', 'panda']:
        animal_dir = os.path.join(base_dir, 'animals', animal_type)
        images = os.listdir(animal_dir)

        # Split images into train and validation (e.g., 80% train, 20% validation)
        split_index = int(0.8 * len(images))
        train_images = images[:split_index]
        val_images = images[split_index:]

        # Move images to respective directories
        for image in train_images:
            src = os.path.join(animal_dir, image)
            dst = os.path.join(train_dir, animal_type, image)
            os.makedirs(os.path.dirname(dst), exist_ok=True) # create subdir if needed
            shutil.copy(src, dst)

        for image in val_images:
            src = os.path.join(animal_dir, image)
            dst = os.path.join(val_dir, animal_type, image)
            os.makedirs(os.path.dirname(dst), exist_ok=True) # create subdir if needed
            shutil.copy(src, dst)

    # ... (continue with your code to create generators, model, training, etc.) ...

    # Data Preprocessing
    train_datagen = ImageDataGenerator(
        rescale=1.0 / 255,
        rotation_range=20,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        fill_mode='nearest'
    )
    val_datagen = ImageDataGenerator(rescale=1.0 / 255)

    train_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=(224, 224),
        batch_size=32,
        class_mode='categorical'
    )

    val_generator = val_datagen.flow_from_directory(
        val_dir,
        target_size=(224, 224),
        batch_size=32,
        class_mode='categorical'
    )

    # Load Pre-trained Model
    base_model = MobileNetV2(weights='imagenet', include_top=False)

    # Add Custom Layers
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    predictions = Dense(len(train_generator.class_indices), activation='softmax')(x)

    # Create Final Model
    model = Model(inputs=base_model.input, outputs=predictions)

    # Freeze Base Model Layers
    for layer in base_model.layers:
        layer.trainable = False

    # Compile the Model
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    # Train the Model
    model.fit(
        train_generator,
        epochs=10,
        validation_data=val_generator
    )

    # Save the Model
    model.save("transfer_learning_model.h5")

    print("Model training complete and saved as transfer_learning_model.h5")

if __name__ == "__main__":
    dataset_path = "/content/animal_dataset"  # Replace with your dataset path
    transfer_learning(dataset_path)

Found 2400 images belonging to 3 classes.
Found 600 images belonging to 3 classes.


<ipython-input-5-c06a43a676ce>:70: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(weights='imagenet', include_top=False)


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Epoch 1/10


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


75/75 ━━━━━━━━━━━━━━━━━━━━ 51s 495ms/step - accuracy: 0.8816 - loss: 0.2763 - val_accuracy: 0.9683 - val_loss: 0.0713
Epoch 2/10
75/75 ━━━━━━━━━━━━━━━━━━━━ 33s 409ms/step - accuracy: 0.9764 - loss: 0.0685 - val_accuracy: 0.9733 - val_loss: 0.0691
Epoch 3/10
75/75 ━━━━━━━━━━━━━━━━━━━━ 33s 406ms/step - accuracy: 0.9785 - loss: 0.0584 - val_accuracy: 0.9800 - val_loss: 0.0632
Epoch 4/10
75/75 ━━━━━━━━━━━━━━━━━━━━ 40s 393ms/step - accuracy: 0.9820 - loss: 0.0473 - val_accuracy: 0.9767 - val_loss: 0.0682
Epoch 5/10
75/75 ━━━━━━━━━━━━━━━━━━━━ 41s 399ms/step - accuracy: 0.9842 - loss: 0.0393 - val_accuracy: 0.9717 - val_loss: 0.0866
Epoch 6/10
75/75 ━━━━━━━━━━━━━━━━━━━━ 42s 410ms/step - accuracy: 0.9819 - loss: 0.0538 - val_accuracy: 0.9733 - val_loss: 0.0756
Epoch 7/10
75/75 ━━━━━━━━━━━━━━━━━━━━ 33s 405ms/step - accuracy: 0.9892 - loss: 0.0301 - val_accuracy: 0.9633 - val_loss: 0.1056
Epoch 8/10
75/75 ━━━━━━━━━━━━━━━━━━━━ 40s 396ms/step - accuracy: 0.9758 - loss: 0.0468 - val_accuracy: 0.973

Model training complete and saved as transfer_learning_model.h5


# **TESTING USING TRANVERSE LEARNING**

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

def test_model(base_dir):
    # Load the trained model
    model = tf.keras.models.load_model("transfer_learning_model.h5")
    print("Model loaded successfully.")

    # Test data directory
    test_dir = os.path.join(base_dir, 'validation')

    # Ensure the test directory exists
    if not os.path.exists(test_dir):
        raise FileNotFoundError(f"Test directory not found: {test_dir}")

    # Preprocess the test data
    test_datagen = ImageDataGenerator(rescale=1.0 / 255)

    test_generator = test_datagen.flow_from_directory(
        test_dir,
        target_size=(224, 224),
        batch_size=32,
        class_mode='categorical',
        shuffle=False  # Do not shuffle for consistent evaluation
    )

    # Evaluate the model on the test data
    test_loss, test_accuracy = model.evaluate(test_generator)
    print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
    print(f"Test Loss: {test_loss:.4f}")

if __name__ == "__main__":
    dataset_path = "/content/animal_dataset"  # Replace with your dataset path
    test_model(dataset_path)


Model loaded successfully.
Found 600 images belonging to 3 classes.
19/19 ━━━━━━━━━━━━━━━━━━━━ 6s 163ms/step - accuracy: 0.9738 - loss: 0.1012
Test Accuracy: 97.17%
Test Loss: 0.1063


***Question 2***

In [3]:
import os
import cv2
import numpy as np
from deep_sort_realtime.deepsort_tracker import DeepSort
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# Load pre-trained model for feature extraction
feature_extractor = MobileNetV2(weights="imagenet", include_top=False, pooling="avg")

# Initialize DeepSORT Tracker
tracker = DeepSort(max_age=30, nn_budget=70, override_track_class=None)

# Video input/output setup
input_video_path = "input_video.mp4"  # Replace with the path to your video
output_video_path = "output_tracked_video.mp4"
cap = cv2.VideoCapture(input_video_path)

# Video writer setup
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter(output_video_path, fourcc, 30.0, (int(cap.get(3)), int(cap.get(4))))

# Process video frame-by-frame
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Prepare detections (simulate bounding boxes, class IDs, and confidence for this example)
    # In a real-world application, you would use an object detector like YOLO or SSD.
    height, width, _ = frame.shape
    dummy_boxes = np.array([[50, 50, 200, 200], [300, 100, 450, 250]])  # Example bounding boxes
    dummy_scores = [0.9, 0.85]  # Confidence scores
    dummy_classes = [1, 2]  # Dummy class IDs (e.g., 1 for dog, 2 for cat)

    # Extract features for each detection
    features = []
    for box in dummy_boxes:
        x1, y1, x2, y2 = box.astype(int)
        cropped_img = frame[y1:y2, x1:x2]
        cropped_img = cv2.resize(cropped_img, (224, 224))
        img_array = img_to_array(cropped_img)
        img_array = preprocess_input(img_array)
        feature = feature_extractor.predict(np.expand_dims(img_array, axis=0))
        features.append(feature.flatten())

    # Update tracker with detections
    tracked_objects = tracker.update_tracks(
        dummy_boxes, dummy_scores, dummy_classes, frame, features
    )

    # Draw tracking results on frame
    for track in tracked_objects:
        if not track.is_confirmed() or track.time_since_update > 1:
            continue

        track_id = track.track_id
        ltrb = track.to_ltrb()
        x1, y1, x2, y2 = map(int, ltrb)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, f"ID: {track_id}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Write the processed frame to output video
    out.write(frame)

    # Optionally display the frame (press 'q' to quit)
    cv2.imshow('Tracked Video', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release resources
cap.release()
out.release()
cv2.destroyAllWindows()

print(f"Tracking complete. Output saved to {output_video_path}")


<ipython-input-3-bf21b22c5d43>:10: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  feature_extractor = MobileNetV2(weights="imagenet", include_top=False, pooling="avg")
/usr/local/lib/python3.10/dist-packages/deep_sort_realtime/embedder/embedder_pytorch.py:53: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the us

Tracking complete. Output saved to output_tracked_video.mp4
